In [15]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import time

# Environments
env = gym.make("CartPole-v1")  # training (no rendering)
test_env = gym.make("CartPole-v1", render_mode="human")  # visualization

In [19]:
# Discretization
n_bins = (?, ?, ?, ?)

upper_bounds = [4.8, 5, 0.418, 5]
lower_bounds = [-4.8, -5, -0.418, -5]

def discretize(obs):
    ratios = [(obs[i] - lower_bounds[i]) / (upper_bounds[i] - lower_bounds[i]) for i in range(len(obs))]
    new_obs = [int(round((n_bins[i] - 1) * ratios[i])) for i in range(len(obs))]
    new_obs = [min(n_bins[i] - 1, max(0, new_obs[i])) for i in range(len(obs))]
    return tuple(new_obs)

# Q-table
Q = np.zeros(n_bins + (env.action_space.n,))

# Hyperparameters
alpha = ?
gamma = ?
epsilon = ?
epsilon_decay = ?
epsilon_min = ?

def choose_action(state):
    if np.random.random() < epsilon:
        return env.action_space.sample()
    return np.argmax(Q[state])

def update_q(state, action, reward, new_state):
    best_next = np.max(Q[new_state])
    Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])

In [ ]:
episodes = ?
rewards = []

for ep in range(episodes):
    obs, _ = env.reset()
    state = discretize(obs)
    total_reward = 0
    done = False

    while not done:
        action = choose_action(state)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        new_state = discretize(obs)

        update_q(state, action, reward, new_state)
        state = new_state
        total_reward += reward

    rewards.append(total_reward)

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Progress print
    if ep % 50 == 0:
        print(f"Episode {ep}, Reward: {total_reward}, Epsilon: {epsilon:.6f}")

# Plot learning curve
plt.plot(rewards)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Training Progress")
plt.show()

In [ ]:
print("Watching trained agent...\n")

for episode in range(5):
    print(f"Test Episode {episode+1}")
    
    obs, _ = test_env.reset()
    done = False
    total_reward = 0

    while not done:
        state = discretize(obs)
        action = np.argmax(Q[state])  # greedy policy (no exploration)
        obs, reward, terminated, truncated, _ = test_env.step(action)
        done = terminated or truncated
        
        total_reward += reward
        time.sleep(0.02)

    print(f"Reward: {total_reward}\n")
    time.sleep(1)

In [22]:
# Close the environment when finished
env.close()
test_env.close()